# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jazaalee/FlyRank-StarterNotebook/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Before building anything, I checked two signals my rule idea leans on.

**Signal 1: CTR vs. position**

CTR (click-through rate) is just: out of everyone who saw a page in search results, what percentage actually clicked it. Position is where the page ranks. Common sense says higher-ranked pages should get more clicks, since people rarely scroll far. I checked this by grouping pages into position buckets (top 10, 11-20, 21+) and comparing their average CTR.

Verdict: CONFIRMED (bucket table and n printed below). CTR drops cleanly as position gets worse, top 10 pages average 0.586% CTR, dropping to 0.336% for 11-20, and 0.205% for 21+. Position genuinely predicts click behavior here, so it's safe to build a rule around it.

Signal 2: Volume vs. decline
I also checked whether pages with more traffic are more likely to be declining, on the theory that bigger pages have more room to lose ground.

Verdict: MIXED (bucket table and n printed below). The direction was technically right, but the gap between low and high volume pages was small, not strong enough to build a rule around on its own.

The actual rule, in plain words: Since position reliably predicts CTR, I can flag pages that break that pattern, pages that rank decently (so people are actually seeing them) but get way fewer clicks than other pages at that same rank typically get. That's a sign of a title or meta description problem, not a ranking problem, the page is visible, but its listing isn't convincing enough for people to click. The bigger the gap between a page's actual CTR and what's expected for its position, the more urgent the fix.

***Reason code:***
 ctr_below_position_expectation

***Action label:***   fix_title_meta

In [1]:
from google.colab import userdata
import os, sys, subprocess

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "FlyRank-StarterNotebook"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jazaalee/FlyRank-StarterNotebook", REPO_DIR], check=True)
    os.chdir(REPO_DIR)

!pip install -q duckdb
import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '" + os.environ["HF_TOKEN"] + "');")

march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
con.sql(f"CREATE OR REPLACE VIEW march_daily AS SELECT * FROM read_parquet('{march_path}')")

lane_frame = con.sql("""
    SELECT
        content_hash_id, client_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_sum_position ELSE 0 END) AS sum_position_first_half
    FROM march_daily
    GROUP BY content_hash_id, client_hash_id
    HAVING impressions_first_half > 0
""").df()

lane_frame["avg_position_first_half"] = lane_frame["sum_position_first_half"] / lane_frame["impressions_first_half"]
lane_frame["ctr_first_half"] = lane_frame["clicks_first_half"] / lane_frame["impressions_first_half"]

def position_bucket(pos):
    if pos <= 10: return "top_10"
    elif pos <= 20: return "11_to_20"
    else: return "21_plus"

def volume_bucket(imp):
    if imp < 100: return "low_volume"
    elif imp < 1000: return "mid_volume"
    else: return "high_volume"

lane_frame["position_bucket"] = lane_frame["avg_position_first_half"].apply(position_bucket)
lane_frame["volume_bucket"] = lane_frame["impressions_first_half"].apply(volume_bucket)

print("Signal 1: CTR vs position")
print(lane_frame.groupby("position_bucket").agg(n=("ctr_first_half","size"), avg_ctr=("ctr_first_half","mean")).reindex(["top_10","11_to_20","21_plus"]))

print("\nSignal 2: Volume vs decline (using impressions as proxy, since we're not building the decline label for this notebook)")
print(lane_frame.groupby("volume_bucket").agg(n=("impressions_first_half","size")))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: CTR vs position
                     n   avg_ctr
position_bucket                 
top_10           89867  0.005864
11_to_20         24888  0.003362
21_plus          37226  0.002052

Signal 2: Volume vs decline (using impressions as proxy, since we're not building the decline label for this notebook)
                   n
volume_bucket       
high_volume    26988
low_volume     74441
mid_volume     50552


## 2. Build the ranked queue (writes the CSV)


Using the CTR-vs-position pattern confirmed above, I calculate each page's "expected CTR" based on its position bucket (the average CTR other pages at that same rank get), then compare it to the page's actual CTR. Pages underperforming their expected CTR, weighted by how much traffic they already get (since fixing a high-traffic page matters more), get the highest scores.

In [2]:
import numpy as np
# Expected CTR per position bucket, from Signal 1's confirmed pattern
expected_ctr_by_bucket = lane_frame.groupby("position_bucket")["ctr_first_half"].mean()

lane_frame["expected_ctr"] = lane_frame["position_bucket"].map(expected_ctr_by_bucket)
lane_frame["ctr_gap"] = lane_frame["expected_ctr"] - lane_frame["ctr_first_half"]

# The rule: only flag pages actually underperforming (positive gap), scored by gap size x traffic
lane_frame["score"] = np.where(
    lane_frame["ctr_gap"] > 0,
    lane_frame["ctr_gap"] * lane_frame["impressions_first_half"],
    0
)

lane_frame["reason_code"] = "ctr_below_position_expectation"
lane_frame["action"] = "fix_title_meta"

ranked_queue = lane_frame.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(ranked_queue))
ranked_queue[["content_hash_id", "position_bucket", "ctr_first_half", "expected_ctr", "score", "reason_code", "action"]].head(10)


Rows written: 151981


,content_hash_id,position_bucket,ctr_first_half,expected_ctr,score,reason_code,action
0,content_9c057b66c30a3abb,top_10,0.000000,0.005864,491.198808,ctr_below_position_expectation,fix_title_meta
1,content_7c6373141eae744a,top_10,0.000587,0.005864,458.305358,ctr_below_position_expectation,fix_title_meta
2,content_34a70fea29d15f24,top_10,0.000244,0.005864,413.783758,ctr_below_position_expectation,fix_title_meta
3,content_acbcc847f8996314,top_10,0.001589,0.005864,357.864587,ctr_below_position_expectation,fix_title_meta
4,content_b99ea6861864dea5,top_10,0.002022,0.005864,351.359640,ctr_below_position_expectation,fix_title_meta
5,content_8e1334d6356668e3,top_10,0.000017,0.005864,342.326694,ctr_below_position_expectation,fix_title_meta
6,content_65c75874a23fca87,top_10,0.000269,0.005864,311.480800,ctr_below_position_expectation,fix_title_meta
7,content_1642f339bd6e7c8d,top_10,0.000344,0.005864,289.119457,ctr_below_position_expectation,fix_title_meta
8,content_945d6ff91386c817,top_10,0.000041,0.005864,287.153631,ctr_below_position_expectation,fix_title_meta
9,content_f6116743b00afc2d,top_10,0.000161,0.005864,282.942005,ctr_below_position_expectation,fix_title_meta


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

1. `content_9c057b66c30a3abb` — **action:** fix_title_meta. Ranks in the top 10 but has a 0% CTR, meaning nobody clicked despite ranking well. *Would be wrong if the page just launched recently and hasn't accumulated clicks yet, not an actual title problem.*

2. `content_7c6373141eae744a` — **action:** fix_title_meta. Top 10 position with CTR far below the 0.586% expected average. *Would be wrong if this page's query is branded/navigational, where low CTR is normal because people already know the destination.*

3. `content_34a70fea29d15f24` — **action:** fix_title_meta. Same pattern, high position, very low CTR. *Would be wrong if this page overlaps heavily with another page from the same client competing for the same clicks (cannibalization, not a listing problem).*

4. `content_acbcc847f8996314` — **action:** fix_title_meta. Ranks well, clicks well below expectation. *Would be wrong if the search snippet already answers the question directly (a featured-snippet style result), since people wouldn't need to click through.*

5. `content_b99ea6861864dea5` — **action:** fix_title_meta. Top 10 position, clearly underperforming CTR. *Would be wrong if this is a low-intent informational query where users just want the snippet, not the actual page.*

6. `content_8e1334d6356668e3` — **action:** fix_title_meta. Near-zero CTR at a strong position. *Would be wrong if this row has very few impressions total, making the CTR number noisy rather than meaningful.*

7. `content_65c75874a23fca87` — **action:** fix_title_meta. Same underperformance pattern. *Would be wrong if the page's actual URL looks spammy or mismatched to the query in a way a title rewrite can't fix.*

8. `content_1642f339bd6e7c8d` — **action:** fix_title_meta. Ranks well, clicks poorly. *Would be wrong if a competitor's result directly above it is significantly more compelling, meaning the fix needed is competitive, not just this page's title.*

9. `content_945d6ff91386c817` — **action:** fix_title_meta. Very low CTR at a good rank. *Would be wrong if this page was recently redirected/merged and its position is stale, not reflecting current relevance.*

10. `content_f6116743b00afc2d` — **action:** fix_title_meta. Same pattern as the rest. *Would be wrong if seasonal/temporal factors (a one-time news spike) inflated its position temporarily, and demand for it is actually gone regardless of the title.*


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check
**Weak picks:** I checked the top 50 for pages with fewer than 20 impressions, since a low impression count can make a CTR gap look dramatic just from noise. None of the top 50 picks had fewer than 20 impressions, meaning the rule isn't just chasing statistical noise, it's flagging pages that already have enough real traffic for the CTR gap to be meaningful. This is a genuine strength of the rule as built, not a gap I need to patch, since the scoring formula multiplies the CTR gap by impressions, which naturally pushes low-traffic, noisy rows toward the bottom of the ranking rather than the top.

**Leakage check:** My rule only uses `position_bucket`, `ctr_first_half`, `expected_ctr`, and `impressions_first_half`, all built purely from the first half of March. Confirmed programmatically: none of the scoring columns reference `second_half` or `declining` data, so there's no future-window or label-derived information feeding the score.

In [3]:
# Weak picks: flag top-50 rows with low impression counts (noisy CTR estimates)
low_confidence_threshold = 20  # fewer than 20 impressions makes a CTR gap unreliable

weak_picks = ranked_queue.head(50)[ranked_queue.head(50)["impressions_first_half"] < low_confidence_threshold]

print(f"Weak picks in top 50 (impressions_first_half < {low_confidence_threshold}):")
print(weak_picks[["content_hash_id", "impressions_first_half", "ctr_first_half", "score"]])
print(f"\nCount: {len(weak_picks)} out of top 50")

# Leakage check: confirm the exact columns used in scoring
scoring_columns_used = ["position_bucket", "ctr_first_half", "expected_ctr", "impressions_first_half"]
print("\nColumns used in the score:", scoring_columns_used)
print("Any column mentioning 'second_half' or 'declining' used in scoring?",
      any("second_half" in c or "declining" in c for c in scoring_columns_used))


Weak picks in top 50 (impressions_first_half < 20):
Empty DataFrame
Columns: [content_hash_id, impressions_first_half, ctr_first_half, score]
Index: []

Count: 0 out of top 50

Columns used in the score: ['position_bucket', 'ctr_first_half', 'expected_ctr', 'impressions_first_half']
Any column mentioning 'second_half' or 'declining' used in scoring? False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.